In [1]:
from pathlib import Path
import sys
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path('/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR')
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Experimentation with TSFEDL Time-Series Models

This notebook mirrors the experiment-oriented structure of `experiment_pyod_models.ipynb`, but using TSFEDL-based time-series models integrated in RADAR.



## Import Required Libraries

Imports for data loading, preprocessing, window creation, TSFEDL models, metrics, and export of the experiment tables.

In [ ]:
import importlib
import time
from inspect import signature
from pathlib import Path
from statistics import mean

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# --- Non-Forecaster classes (LightningModule) for the direct/base path ---
from TSFEDL.models_pytorch import (
    OhShuLih,
    GaoJunLi,
    KongZhengmin,
    CaiWenjuan,
    WangKejun,
    ZhengZhenyu,
    KimTaeYoung,
    FuJiangmeng,
    ShiHaotian,
    SharPar,
    HongTan,
    HtetMyetLynn,
    LihOhShu,
    YiboGao,
    YaoQihang,
    YildirimOzal,
    ZhangJin,
    WeiXiaoyan,
    KhanZulfiqar,
    ChenChen,
    GenMinxing,
    HuangMeiLing,
    DaiXiLi,
    TSFEDL_BaseModule,
)
# --- Forecaster classes (nn.Module) used as top_module ---
from TSFEDL.models_pytorch import (
    CaiWenjuan_Forecaster,
    ChenChen_Forecaster,
    DaiXiLi_Forecaster,
    FuJiangmeng_Forecaster,
    GaoJunLi_Forecaster,
    GenMinxing_Forecaster,
    HongTan_Forecaster,
    HtetMyetLynn_Forecaster,
    HuangMeiLing_Forecaster,
    KhanZulfiqar_Forecaster,
    KimTaeYoung_Forecaster,
    KongZhengmin_Forecaster,
    LihOhShu_Forecaster,
    OhShuLih_Forecaster,
    SharPar_Forecaster,
    ShiHaotian_Forecaster,
    WangKejun_Forecaster,
    WeiXiaoyan_Forecaster,
    YaoQihang_Forecaster,
    YiboGao_Forecaster,
    YildirimOzal_Forecaster,
    ZhangJin_Forecaster,
    ZhengZhenyu_Forecaster,
)
from RADAR.time_series.algorithms import tsfedl
from RADAR.time_series.preprocessing.preprocessing_ts import StandardScalerPreprocessing
from RADAR.time_series.time_series_datasets_uci import global_load as load_time_series
from RADAR.time_series.time_series_utils import TimeSeriesProcessor
import RADAR.metrics_module as metrics_module

metrics_module = importlib.reload(metrics_module)

2026-03-13 21:45:17.162442: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-13 21:45:17.176018: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773434717.192496  356609 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773434717.196402  356609 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-13 21:45:17.212837: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

## Build the UCI Time-Series Benchmark

The preprocessing follows the same dataset choices as `test_tsfedl.ipynb` and keeps a chronological split.

- `ai4i_2020_predictive_maintenance_dataset`: labels come from `Machine failure`.
- `metro_interstate_traffic_volume`: anomaly labels are derived from extreme traffic levels using the 5th and 95th percentiles.

To keep the benchmark aligned with anomaly detection, the TSFEDL models are trained in reconstruction mode, using each input window as its own target.

In [4]:
WINDOW_SIZE = 48
STEP_SIZE = 1
TEST_SIZE = 0.2
METRO_LOW_Q = 0.05
METRO_HIGH_Q = 0.95

def chronological_split(X, y, test_size=0.2):
    split_idx = int(len(X) * (1 - test_size))
    return X[:split_idx], X[split_idx:], y[:split_idx], y[split_idx:]

def aggregate_window_labels(y_windows):
    y_windows = np.asarray(y_windows)
    if y_windows.ndim == 1:
        return y_windows.astype(int)
    return (y_windows.sum(axis=1) > 0).astype(int)

def prepare_ai4i_dataset(window_size=WINDOW_SIZE, step_size=STEP_SIZE, test_size=TEST_SIZE):
    X, y = load_time_series('ai4i_2020_predictive_maintenance_dataset')
    labels = y['Machine failure'].astype(int).to_numpy()
    X = X.drop(columns=['Type'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)

    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=False)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows = processor.process_train_test(X_train, y_train, X_test, y_test)

    return {
        'dataset': 'ai4i_2020_predictive_maintenance_dataset',
        'X_train_windows': np.asarray(X_train_windows, dtype=np.float32),
        'X_test_windows': np.asarray(X_test_windows, dtype=np.float32),
        'y_test_labels': aggregate_window_labels(y_test_windows),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'window_size': window_size,
        'train_windows': len(X_train_windows),
        'test_windows': len(X_test_windows),
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'positive_ratio_windows': round(float(np.mean(aggregate_window_labels(y_test_windows))), 4),
        'label_note': 'Machine failure from UCI target',
    }

def prepare_metro_dataset(window_size=WINDOW_SIZE, step_size=STEP_SIZE, test_size=TEST_SIZE, low_q=METRO_LOW_Q, high_q=METRO_HIGH_Q):
    X, y = load_time_series('metro_interstate_traffic_volume')
    traffic_volume = y['traffic_volume'].astype(float)
    low_threshold = float(traffic_volume.quantile(low_q))
    high_threshold = float(traffic_volume.quantile(high_q))
    labels = ((traffic_volume <= low_threshold) | (traffic_volume >= high_threshold)).astype(int).to_numpy()

    X = X.drop(columns=['date_time', 'holiday', 'weather_main', 'weather_description'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)

    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=False)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows = processor.process_train_test(X_train, y_train, X_test, y_test)

    return {
        'dataset': 'metro_interstate_traffic_volume',
        'X_train_windows': np.asarray(X_train_windows, dtype=np.float32),
        'X_test_windows': np.asarray(X_test_windows, dtype=np.float32),
        'y_test_labels': aggregate_window_labels(y_test_windows),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'window_size': window_size,
        'train_windows': len(X_train_windows),
        'test_windows': len(X_test_windows),
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'positive_ratio_windows': round(float(np.mean(aggregate_window_labels(y_test_windows))), 4),
        'label_note': f'Extreme traffic volume: <= q{low_q:.2f} or >= q{high_q:.2f}',
        'low_threshold': round(low_threshold, 3),
        'high_threshold': round(high_threshold, 3),
    }

dataset_configs = {
    'ai4i': prepare_ai4i_dataset(),
    'metro_interstate': prepare_metro_dataset(),
}

Metadata: {'uci_id': 601, 'name': 'AI4I 2020 Predictive Maintenance Dataset', 'repository_url': 'https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/601/data.csv', 'abstract': 'The AI4I 2020 Predictive Maintenance Dataset is a synthetic dataset that reflects real predictive maintenance data encountered in industry.', 'area': 'Computer Science', 'tasks': ['Classification', 'Regression', 'Causal-Discovery'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 10000, 'num_features': 6, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], 'index_col': ['UID', 'Product ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Wed Feb 14 2024', 'dataset_doi': '10.24432/C5HS5C', 'creators': [], 'intro_paper': {'ID': 386, 'type': 'NATIVE', 'title': 'Explainable Artificial 

In [5]:
dataset_summary = pd.DataFrame([
    {
        'dataset_key': dataset_key,
        'dataset_name': config['dataset'],
        'samples': config['n_samples'],
        'features': config['n_features'],
        'window_size': config['window_size'],
        'train_windows': config['train_windows'],
        'test_windows': config['test_windows'],
        'positive_ratio_points': config['positive_ratio_points'],
        'positive_ratio_windows': config['positive_ratio_windows'],
        'label_note': config['label_note'],
    }
    for dataset_key, config in dataset_configs.items()
]).reset_index(drop=True)

display(dataset_summary)

,dataset_key,dataset_name,samples,features,window_size,train_windows,test_windows,positive_ratio_points,positive_ratio_windows,label_note
0,ai4i,ai4i_2020_predictive_maintenance_dataset,10000,5,48,7953,1953,0.0339,0.5832,Machine failure from UCI target
1,metro_interstate,metro_interstate_traffic_volume,48204,4,48,38516,9594,0.1006,0.8919,Extreme traffic volume: <= q0.05 or >= q0.95


## TSFEDL Benchmark on the Two UCI Datasets


In [ ]:
# Minimum number of input features (channels) required by each TSFEDL architecture.
# Models whose kernels exceed the available feature dimension will fail at forward time.
# Values obtained empirically with window_size=48.
MODEL_MIN_FEATURES = {
    'ohshulih': 1,     'gaojunli': 1,     'kongzhengmin': 6,
    'caiwenjuan': 8,   'wangkejun': 8,    'zhengzhenyu': 8,
    'kimtaeyoung': 5,  'fujiangmeng': 2,  'shihaotian': 15,
    'sharpar': 1,      'hongtan': 12,     'htetmyetlynn': 16,
    'liohshu': 261,    'yibogao': 1000,   'yaoqihang': 243,
    'yildirimozal': 8, 'zhangjin': 243,   'weixiaoyan': 96,
    'khanzulfiqar': 261, 'chenchen': 1000, 'genminxing': 1,
    'huangmeiling': 1000, 'daixili': 1000,
}

# Maximum number of features to pad to.  Models that need more than this
# will be skipped rather than artificially inflated.
MAX_PAD_FEATURES = 32


def pad_features(X, target_features):
    """Cyclically repeat features along the last axis until target_features is reached."""
    current = X.shape[-1]
    if current >= target_features:
        return X
    repeats = (target_features // current) + 1
    if isinstance(X, np.ndarray):
        return np.tile(X, (1,) * (X.ndim - 1) + (repeats,))[..., :target_features]
    # torch.Tensor
    return X.repeat(*([1] * (X.ndim - 1)), repeats)[..., :target_features]


def get_forecaster_configs(input_dim, seq_len):
    """Return per-model configuration dictionaries.

    Each entry contains:
      - class         : Forecaster class (nn.Module) used as top_module
      - base_class    : LightningModule class for the direct/base path
      - display_name  : human-readable name
      - min_features  : minimum input features the architecture supports
      - extra_kwargs  : extra keyword args forwarded to the *platform* model params
      - top_module_kwargs : kwargs for constructing the Forecaster (top_module)
      - base_model_kwargs : kwargs for constructing the LightningModule (base path)
                            If empty, defaults to {in_features: seq_len}.
    """
    return {
        'ohshulih': {
            'class': OhShuLih_Forecaster,
            'base_class': OhShuLih,
            'display_name': 'OhShuLih',
            'min_features': MODEL_MIN_FEATURES['ohshulih'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'gaojunli': {
            'class': GaoJunLi_Forecaster,
            'base_class': GaoJunLi,
            'display_name': 'GaoJunLi',
            'min_features': MODEL_MIN_FEATURES['gaojunli'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'kongzhengmin': {
            'class': KongZhengmin_Forecaster,
            'base_class': KongZhengmin,
            'display_name': 'KongZhengmin',
            'min_features': MODEL_MIN_FEATURES['kongzhengmin'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'caiwenjuan': {
            'class': CaiWenjuan_Forecaster,
            'base_class': CaiWenjuan,
            'display_name': 'CaiWenjuan',
            'min_features': MODEL_MIN_FEATURES['caiwenjuan'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'wangkejun': {
            'class': WangKejun_Forecaster,
            'base_class': WangKejun,
            'display_name': 'WangKejun',
            'min_features': MODEL_MIN_FEATURES['wangkejun'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'zhengzhenyu': {
            'class': ZhengZhenyu_Forecaster,
            'base_class': ZhengZhenyu,
            'display_name': 'ZhengZhenyu',
            'min_features': MODEL_MIN_FEATURES['zhengzhenyu'],
            'extra_kwargs': {},
            'top_module_kwargs': {'in_features': 256},
            'base_model_kwargs': {},
        },
        'kimtaeyoung': {
            'class': KimTaeYoung_Forecaster,
            'base_class': KimTaeYoung,
            'display_name': 'KimTaeYoung',
            'min_features': MODEL_MIN_FEATURES['kimtaeyoung'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'fujiangmeng': {
            'class': FuJiangmeng_Forecaster,
            'base_class': FuJiangmeng,
            'display_name': 'FuJiangmeng',
            'min_features': MODEL_MIN_FEATURES['fujiangmeng'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'shihaotian': {
            'class': ShiHaotian_Forecaster,
            'base_class': ShiHaotian,
            'display_name': 'ShiHaotian',
            'min_features': MODEL_MIN_FEATURES['shihaotian'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'sharpar': {
            'class': SharPar_Forecaster,
            'base_class': SharPar,
            'display_name': 'SharPar',
            'min_features': MODEL_MIN_FEATURES['sharpar'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'hongtan': {
            'class': HongTan_Forecaster,
            'base_class': HongTan,
            'display_name': 'HongTan',
            'min_features': MODEL_MIN_FEATURES['hongtan'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'htetmyetlynn': {
            'class': HtetMyetLynn_Forecaster,
            'base_class': HtetMyetLynn,
            'display_name': 'HtetMyetLynn',
            'min_features': MODEL_MIN_FEATURES['htetmyetlynn'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'liohshu': {
            'class': LihOhShu_Forecaster,
            'base_class': LihOhShu,
            'display_name': 'LihOhShu',
            'min_features': MODEL_MIN_FEATURES['liohshu'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'yibogao': {
            'class': YiboGao_Forecaster,
            'base_class': YiboGao,
            'display_name': 'YiboGao',
            'min_features': MODEL_MIN_FEATURES['yibogao'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'yaoqihang': {
            'class': YaoQihang_Forecaster,
            'base_class': YaoQihang,
            'display_name': 'YaoQihang',
            'min_features': MODEL_MIN_FEATURES['yaoqihang'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'yildirimozal': {
            'class': YildirimOzal_Forecaster,
            'base_class': YildirimOzal,
            'display_name': 'YildirimOzal',
            'min_features': MODEL_MIN_FEATURES['yildirimozal'],
            'in_features_value': None,
            # input_shape is a positional param of the LightningModule, NOT the Forecaster
            'extra_kwargs': {'input_shape': (seq_len, input_dim)},
            'top_module_kwargs': {'in_features': input_dim},
            'base_model_kwargs': {'input_shape': (seq_len, input_dim)},
        },
        'zhangjin': {
            'class': ZhangJin_Forecaster,
            'base_class': ZhangJin,
            'display_name': 'ZhangJin',
            'min_features': MODEL_MIN_FEATURES['zhangjin'],
            'extra_kwargs': {},
            'top_module_kwargs': {'in_features': input_dim},
            'base_model_kwargs': {},
        },
        'weixiaoyan': {
            'class': WeiXiaoyan_Forecaster,
            'base_class': WeiXiaoyan,
            'display_name': 'WeiXiaoyan',
            'min_features': MODEL_MIN_FEATURES['weixiaoyan'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'khanzulfiqar': {
            'class': KhanZulfiqar_Forecaster,
            'base_class': KhanZulfiqar,
            'display_name': 'KhanZulfiqar',
            'min_features': MODEL_MIN_FEATURES['khanzulfiqar'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'chenchen': {
            'class': ChenChen_Forecaster,
            'base_class': ChenChen,
            'display_name': 'ChenChen',
            'min_features': MODEL_MIN_FEATURES['chenchen'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'genminxing': {
            'class': GenMinxing_Forecaster,
            'base_class': GenMinxing,
            'display_name': 'GenMinxing',
            'min_features': MODEL_MIN_FEATURES['genminxing'],
            'in_features_value': input_dim,
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'huangmeiling': {
            'class': HuangMeiLing_Forecaster,
            'base_class': HuangMeiLing,
            'display_name': 'HuangMeiLing',
            'min_features': MODEL_MIN_FEATURES['huangmeiling'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
        'daixili': {
            'class': DaiXiLi_Forecaster,
            'base_class': DaiXiLi,
            'display_name': 'DaiXiLi',
            'min_features': MODEL_MIN_FEATURES['daixili'],
            'extra_kwargs': {},
            'top_module_kwargs': {},
            'base_model_kwargs': {},
        },
    }
    }    }

In [7]:
# List of forecasters to benchmark, in desired order
ordered_forecasters = [
    'ohshulih', 'gaojunli', 'kongzhengmin', 'caiwenjuan', 'wangkejun', 'zhengzhenyu',
    'kimtaeyoung', 'fujiangmeng', 'shihaotian', 'sharpar', 'hongtan', 'htetmyetlynn',
    'liohshu', 'yibogao', 'yaoqihang', 'yildirimozal', 'zhangjin', 'weixiaoyan',
    'khanzulfiqar', 'chenchen', 'genminxing', 'huangmeiling', 'daixili'
]


In [8]:
# Number of times to repeat timing for each model (for averaging)
TIMING_REPETITIONS = 1  # You can increase this for more robust timing, e.g., 3 or 5


In [ ]:
BATCH_SIZE = 64
MAX_EPOCHS = 1
TRAINER_DEFAULT_KWARGS = {
    'max_epochs': MAX_EPOCHS,
    'logger': False,
    'enable_checkpointing': False,
    'enable_progress_bar': False,
}


def summarize_mse(scores):
    scores = np.asarray(scores, dtype=float).ravel()
    finite_scores = scores[np.isfinite(scores)]
    return float(np.mean(finite_scores)) if finite_scores.size else np.nan


def build_model_params(forecaster_name, input_dim, seq_len):
    """Build the parameter dict used by both the platform and base paths.

    The ``input_dim`` here is the (possibly padded) feature count so that the
    Forecaster output layer matches the actual tensor width.
    """
    forecaster_config = get_forecaster_configs(input_dim, seq_len)[forecaster_name]
    top_module_kwargs = {
        'out_features': input_dim,
        'n_pred': seq_len,
        **forecaster_config.get('top_module_kwargs', {}),
    }
    top_module = forecaster_config['class'](**top_module_kwargs)

    in_features_val = forecaster_config.get('in_features_value', seq_len)

    result = {
        'algorithm_': forecaster_name,
        'loss': torch.nn.MSELoss(),
        'top_module': top_module,
        'batch_size': BATCH_SIZE,
        **TRAINER_DEFAULT_KWARGS,
        **forecaster_config.get('extra_kwargs', {}),
    }

    if in_features_val is not None:
        result['in_features'] = in_features_val

    return result


def split_model_and_trainer_params(model_params):
    trainer_signature = signature(pl.Trainer.__init__)
    trainer_param_names = {name for name in trainer_signature.parameters if name != 'self'}

    trainer_kwargs = {}
    direct_model_kwargs = {}
    for key, value in model_params.items():
        if key in trainer_param_names:
            trainer_kwargs[key] = value
        else:
            direct_model_kwargs[key] = value

    return direct_model_kwargs, trainer_kwargs


def build_direct_tsfedl_model(forecaster_name, model_kwargs, input_dim, seq_len):
    """Construct the TSFEDL *LightningModule* model for the direct/base path.

    Instead of returning just the Forecaster (which is a plain nn.Module and
    therefore incompatible with ``pl.Trainer``), this builds the full
    TSFEDL_BaseModule subclass with the Forecaster plugged in as ``top_module``.
    """
    config = get_forecaster_configs(input_dim, seq_len)[forecaster_name]
    base_class = config['base_class']
    base_model_extra = config.get('base_model_kwargs', {})

    # Standard constructor kwargs shared by all models
    kwargs = {
        'top_module': model_kwargs['top_module'],
        'loss': model_kwargs['loss'],
    }

    # Most models take ``in_features`` (the sequence length) as positional arg.
    # YildirimOzal instead uses ``input_shape`` – handled via base_model_extra.
    # GenMinxing uses LSTM so in_features = actual feature count (via config).
    if 'input_shape' not in base_model_extra:
        kwargs['in_features'] = config.get('in_features_value', seq_len)

    kwargs.update(base_model_extra)

    return base_class(**kwargs)


def fit_direct_tsfedl_model(direct_model, X_train_tensor, batch_size, trainer_kwargs):
    train_dataset = TensorDataset(X_train_tensor, X_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    trainer = pl.Trainer(**trainer_kwargs)
    trainer.fit(direct_model, train_dataloaders=train_loader)
    return direct_model


def compute_direct_scores(direct_model, X_test_tensor):
    direct_model.eval()
    device = next(direct_model.parameters()).device if any(True for _ in direct_model.parameters()) else torch.device('cpu')
    X_test_tensor = X_test_tensor.to(device)

    with torch.no_grad():
        predictions = direct_model(X_test_tensor)
        if isinstance(predictions, (tuple, list)):
            predictions = predictions[0]

        if predictions.shape == X_test_tensor.shape:
            errors = (predictions - X_test_tensor) ** 2
            reduction_dims = tuple(range(1, errors.ndim))
            return torch.mean(errors, dim=reduction_dims).detach().cpu().numpy()

        predictions_flat = predictions.reshape(predictions.shape[0], -1)
        targets_flat = X_test_tensor.reshape(X_test_tensor.shape[0], -1)
        min_width = min(predictions_flat.shape[1], targets_flat.shape[1])
        errors = (predictions_flat[:, :min_width] - targets_flat[:, :min_width]) ** 2
        return torch.mean(errors, dim=1).detach().cpu().numpy()

In [ ]:
tsfedl_results = []

for dataset_key, config in dataset_configs.items():
    raw_input_dim = config['X_train_windows'].shape[2]  # original feature count
    seq_len = config['window_size']

    print(f'\nDataset: {config["dataset"]}')
    print(f'Features: {raw_input_dim} | Train windows: {config["train_windows"]} | Test windows: {config["test_windows"]}')

    for model_order, forecaster_name in enumerate(ordered_forecasters, start=1):
        # --- Determine effective feature count (with optional padding) ---
        min_feat = MODEL_MIN_FEATURES.get(forecaster_name, 1)
        if min_feat > MAX_PAD_FEATURES:
            effective_dim = raw_input_dim  # will likely fail; recorded as architecture skip
        else:
            effective_dim = max(raw_input_dim, min_feat)

        need_padding = effective_dim > raw_input_dim
        skip_model = (min_feat > MAX_PAD_FEATURES) and (raw_input_dim < min_feat)

        # Build (potentially padded) tensors
        if need_padding and not skip_model:
            X_train_tensor = pad_features(
                torch.tensor(config['X_train_windows'], dtype=torch.float32),
                effective_dim,
            )
            X_test_tensor = pad_features(
                torch.tensor(config['X_test_windows'], dtype=torch.float32),
                effective_dim,
            )
        else:
            X_train_tensor = torch.tensor(config['X_train_windows'], dtype=torch.float32)
            X_test_tensor = torch.tensor(config['X_test_windows'], dtype=torch.float32)

        FORECASTER_CONFIGS = get_forecaster_configs(effective_dim, seq_len)
        display_name = FORECASTER_CONFIGS[forecaster_name]['display_name']
        pad_note = f' (padded {raw_input_dim}→{effective_dim} features)' if need_padding and not skip_model else ''
        print(f'  [{model_order:02d}] Model: {display_name}{pad_note}')

        result_row = {
            'dataset_key': dataset_key,
            'dataset_name': config['dataset'],
            'model_order': model_order,
            'model_display_name': display_name,
            'algorithm': forecaster_name,
            'window_size': seq_len,
            'n_features': raw_input_dim,
            'effective_features': effective_dim,
            'padded': need_padding and not skip_model,
            'train_windows': config['train_windows'],
            'test_windows': config['test_windows'],
            'timing_repetitions': TIMING_REPETITIONS,
            'platform_status': 'pending',
            'base_status': 'pending',
            'platform_error_message': None,
            'base_error_message': None,
            'average_platform_time_s': np.nan,
            'average_base_time_s': np.nan,
            'overhead_s': np.nan,
            'speedup_base_over_platform': np.nan,
            'platform_mse': np.nan,
            'base_mse': np.nan,
            'mse_diff': np.nan,
        }

        # ---- Skip models whose architecture is incompatible ----
        if skip_model:
            skip_msg = (
                f'Requires >= {min_feat} input features, dataset has {raw_input_dim} '
                f'(exceeds MAX_PAD_FEATURES={MAX_PAD_FEATURES})'
            )
            result_row['platform_status'] = 'skipped'
            result_row['base_status'] = 'skipped'
            result_row['platform_error_message'] = skip_msg
            result_row['base_error_message'] = skip_msg
            print(f'    Skipped: {skip_msg}')
            tsfedl_results.append(result_row)
            continue

        # ============================================================
        # Platform path (RADAR wrapper)
        # ============================================================
        try:
            platform_execution_times = []
            platform_model = None
            for _ in tqdm(
                range(TIMING_REPETITIONS),
                desc=f'Platform Timing ({display_name} | {dataset_key})',
                leave=False,
            ):
                platform_params = build_model_params(forecaster_name, effective_dim, seq_len)
                platform_model = tsfedl.TsfedlAnomalyDetection(**platform_params)
                start_time = time.time()
                platform_model.fit(
                    config['X_train_windows'] if not (need_padding and not skip_model)
                    else pad_features(np.asarray(config['X_train_windows'], dtype=np.float32), effective_dim),
                    X_train_tensor,
                )
                platform_execution_times.append(time.time() - start_time)

            average_platform_time = mean(platform_execution_times)
            platform_test_data = (
                config['X_test_windows'] if not (need_padding and not skip_model)
                else pad_features(np.asarray(config['X_test_windows'], dtype=np.float32), effective_dim)
            )
            platform_scores = np.asarray(platform_model.decision_function(platform_test_data)).ravel()
            finite_platform_scores = bool(np.isfinite(platform_scores).all())

            result_row.update({
                'platform_status': 'ok',
                'average_platform_time_s': round(average_platform_time, 4),
                'platform_mse': round(summarize_mse(platform_scores), 6) if finite_platform_scores else np.nan,
            })

            print(
                f'    Platform -> time={average_platform_time:.4f}s | '
                f'MSE={result_row["platform_mse"]:.6f}'
            )
        except Exception as exc:
            result_row['platform_status'] = 'failed'
            result_row['platform_error_message'] = str(exc).strip() or exc.__class__.__name__
            print(f'    Platform failed: {result_row["platform_error_message"]}')

        # ============================================================
        # Direct / Base path (TSFEDL LightningModule without RADAR wrapper)
        # ============================================================
        try:
            direct_execution_times = []
            direct_model = None
            for _ in tqdm(
                range(TIMING_REPETITIONS),
                desc=f'Direct TSFEDL Timing ({display_name} | {dataset_key})',
                leave=False,
            ):
                platform_params = build_model_params(forecaster_name, effective_dim, seq_len)
                model_kwargs, trainer_kwargs = split_model_and_trainer_params(platform_params)
                # Build the proper LightningModule (not just the Forecaster)
                direct_model = build_direct_tsfedl_model(
                    forecaster_name, model_kwargs, effective_dim, seq_len,
                )
                start_time = time.time()
                fit_direct_tsfedl_model(
                    direct_model=direct_model,
                    X_train_tensor=X_train_tensor,
                    batch_size=platform_params['batch_size'],
                    trainer_kwargs=trainer_kwargs,
                )
                direct_execution_times.append(time.time() - start_time)

            average_base_time = mean(direct_execution_times)
            direct_scores = compute_direct_scores(direct_model, X_test_tensor)
            finite_direct_scores = bool(np.isfinite(direct_scores).all())

            result_row.update({
                'base_status': 'ok',
                'average_base_time_s': round(average_base_time, 4),
                'base_mse': round(summarize_mse(direct_scores), 6) if finite_direct_scores else np.nan,
            })

            print(f'    Direct TSFEDL -> time={average_base_time:.4f}s | MSE={result_row["base_mse"]:.6f}')
        except Exception as exc:
            result_row['base_status'] = 'failed'
            result_row['base_error_message'] = str(exc).strip() or exc.__class__.__name__
            print(f'    Direct TSFEDL failed: {result_row["base_error_message"]}')

        if np.isfinite(result_row['average_platform_time_s']) and np.isfinite(result_row['average_base_time_s']):
            result_row['overhead_s'] = round(
                result_row['average_platform_time_s'] - result_row['average_base_time_s'],
                4,
            )
            result_row['speedup_base_over_platform'] = round(
                result_row['average_base_time_s'] / result_row['average_platform_time_s'],
                4,
            ) if result_row['average_platform_time_s'] > 0 else np.nan

        if np.isfinite(result_row['platform_mse']) and np.isfinite(result_row['base_mse']):
            result_row['mse_diff'] = round(
                result_row['platform_mse'] - result_row['base_mse'],
                6,
            )

        tsfedl_results.append(result_row)

tsfedl_results_df = pd.DataFrame(tsfedl_results).sort_values(
    ['dataset_name', 'model_order'],
    ascending=[True, True],
).reset_index(drop=True)

display(tsfedl_results_df)

## Per-Dataset Summary

This table compares the RADAR execution path with the direct TSFEDL library execution path and highlights the resulting overhead together with the reconstruction error reported as MSE.

In [ ]:
tsfedl_summary_df = tsfedl_results_df[
    [
        'dataset_name',
        'model_order',
        'model_display_name',
        'algorithm',
        'n_features',
        'effective_features',
        'padded',
        'platform_status',
        'base_status',
        'average_platform_time_s',
        'average_base_time_s',
        'overhead_s',
        'speedup_base_over_platform',
        'platform_mse',
        'base_mse',
        'mse_diff',
        'platform_error_message',
        'base_error_message',
    ]
].copy()

display(tsfedl_summary_df)

In [ ]:
results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

results_main_path = results_dir / 'uci_tsfedl_results.csv'
results_summary_path = results_dir / 'uci_tsfedl_summary.csv'

tsfedl_results_df.to_csv(results_main_path, index=False)
tsfedl_summary_df.to_csv(results_summary_path, index=False)

print(f'Saved detailed results to: {results_main_path}')
print(f'Saved summary results to: {results_summary_path}')

Saved detailed results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_tsfedl_results.csv
Saved summary results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_tsfedl_summary.csv
